In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree


# 1. Load the datasets
collision_df = pd.read_csv('final_mapped_accidents_with_distances.csv')
collision_df = collision_df.drop(columns=['year'])
mapping_df = pd.read_csv('metadata_images_mapping.csv')
features_df = pd.read_csv('processed_features_conf_0.5_probs.csv')

# Clean missing values
mapping_df = mapping_df.dropna(subset=['image_blob_name', 'lat', 'lng'])
features_df = features_df.dropna(subset=['image_name'])
features_df = features_df.fillna(0)
# 2. Link Features to Coordinates
# Extract the base filename (e.g., '4411_0.jpg') from the full blob path
mapping_df['image_filename'] = mapping_df['image_blob_name'].astype(str).apply(lambda x: x.split('/')[-1])

# Merge visual features with their lat/lng coordinates
visual_data = pd.merge(
    features_df,
    mapping_df[['image_filename', 'lat', 'lng', 'year']],
    left_on='image_name',
    right_on='image_filename',
    how='inner'
)

# Average the features for the 4 image angles at each unique location
visual_features_per_loc = visual_data.drop(columns=['image_name', 'image_filename']).groupby(['lat', 'lng']).mean().reset_index()

# 3. Spatial Search (100m Radius)
# We scale Longitude by cos(Lat) to approximate Euclidean distance in meters
# If your study area isn't London, change mean_lat accordingly.
mean_lat = np.radians(51.5)  # Approximate latitude for London
lng_scale = np.cos(mean_lat)

visual_coords = visual_features_per_loc[['lat', 'lng']].values
visual_coords_scaled = visual_coords.copy()
visual_coords_scaled[:, 1] *= lng_scale

collision_coords = collision_df[['latitude', 'longitude']].values
collision_coords_scaled = collision_coords.copy()
collision_coords_scaled[:, 1] *= lng_scale

# Build spatial index
tree = cKDTree(visual_coords_scaled)

# 500 meters expressed in degrees latitude
radius_deg = 500 / 111320.0

# Find the nearest visual features within 500m for every accident
distances, indices = tree.query(collision_coords_scaled, distance_upper_bound=radius_deg)

# 4. Merge Data (only keep matched points)
matched_mask = indices < len(visual_features_per_loc)
feature_cols = visual_features_per_loc.columns.drop(['lat', 'lng'])

# Prepare empty columns for features
matched_feature_values = np.full((len(collision_df), len(feature_cols)), np.nan)

# Fill rows where a match was found
if matched_mask.any():
    matched_feature_values[matched_mask] = visual_features_per_loc.iloc[indices[matched_mask]][feature_cols].values

# Create final DataFrame (enriched)
features_to_add = pd.DataFrame(matched_feature_values, columns=feature_cols, index=collision_df.index)
enriched_df = pd.concat([collision_df, features_to_add], axis=1)

total_collisions = len(collision_df)
matched_count = int(matched_mask.sum())
matched_pct = matched_mask.mean() * 100.0

print(f"Total collisions (input): {total_collisions}")
print(f"Matched with features: {matched_count} ({matched_pct:.2f}%)")

# Only keep matched rows for final CSV (user requested: don't include unmatched points)
if matched_count == 0:
    print("No collisions were matched to visual features within 500m. No output CSV will be written.")
else:
    matched_df = enriched_df.loc[matched_mask].copy()  # only rows that had a match

    # --- Identify the severity column automatically (common names) ---
    severity_candidates = [
        'accident_severity', 'severity', 'Accident_Severity', 'severity_label',
        'acc_sev', 'severity_code', 'severity_type'
    ]
    severity_col = None
    # prefer exact matches first
    for c in severity_candidates:
        if c in matched_df.columns:
            severity_col = c
            break

    # fallback: look for a column containing 'sever' or 'accident' in its name
    if severity_col is None:
        for c in matched_df.columns:
            if 'sever' in c.lower() or 'accident' in c.lower():
                severity_col = c
                break

    # If still None, we won't compute severity stats (but will save CSV)
    if severity_col is None:
        print("Warning: Could not automatically find a severity column in the matched data.")
        print("Available columns (sample):", list(matched_df.columns[:30]))
        # Save matched only
        matched_df.to_csv('enriched_collision_data_500m_bad.csv', index=False)
        print("Saved matched data to 'enriched_collision_data_500m_bad.csv' (no severity stats).")
    else:
        print(f"Detected severity column: '{severity_col}'")

        # Normalize / map severity to canonical labels: 'Fatal', 'Serious', 'Slight'
        sev_series = matched_df[severity_col]

        def normalize_severity(s):
            # handle numeric encodings (common: 1=Fatal,2=Serious,3=Slight)
            if pd.isna(s):
                return np.nan
            # numeric-like
            if isinstance(s, (int, np.integer, float, np.floating)):
                try:
                    si = int(s)
                    if si == 1:
                        return 'Fatal'
                    elif si == 2:
                        return 'Serious'
                    elif si == 3:
                        return 'Slight'
                except Exception:
                    pass
            # strings: check keywords
            s_str = str(s).strip().lower()
            if 'fatal' in s_str or s_str == '1':
                return 'Fatal'
            if 'serious' in s_str or 'serious-injury' in s_str or s_str == '2':
                return 'Serious'
            if 'slight' in s_str or 'minor' in s_str or s_str == '3':
                return 'Slight'
            # other common encodings: "KSI" or "S" etc are ambiguous; return raw
            return s_str  # leave as-is for later counting

        matched_df['accident_severity_normalized'] = sev_series.map(normalize_severity)

        # Count canonical categories (ensure the requested order)
        categories = ['Fatal', 'Serious', 'Slight']
        counts = matched_df['accident_severity_normalized'].value_counts(dropna=False)

        # Safely get counts for our three categories (0 if missing)
        fatal_count = int(counts.get('Fatal', 0))
        serious_count = int(counts.get('Serious', 0))
        slight_count = int(counts.get('Slight', 0))

        # Percentages relative to matched_count
        def pct(n):
            return (n / matched_count * 100.0) if matched_count > 0 else 0.0

        fatal_pct = pct(fatal_count)
        serious_pct = pct(serious_count)
        slight_pct = pct(slight_count)

        print("\nMatched set severity breakdown (counts and % of matched set):")
        print(f"  Fatal:   {fatal_count} ({fatal_pct:.2f}%)")
        print(f"  Serious: {serious_count} ({serious_pct:.2f}%)")
        print(f"  Slight:  {slight_count} ({slight_pct:.2f}%)")

        # Optionally: show any other (unexpected) severity labels found
        other_labels = [lab for lab in counts.index if lab not in categories and pd.notna(lab)]
        if other_labels:
            print("\nOther / unrecognized severity labels present in matched data (counts):")
            for lab in other_labels:
                print(f"  {lab}: {int(counts[lab])}")
        matched_df.rename(columns={'year': 'infrastructure_year'})

        # Save matched only (drop rows where no match)
        # Keep the normalized severity column so you can inspect later
        matched_df.to_csv('enriched_collision_data_500m_new.csv', index=False)
        print("\nSaved matched data to 'enriched_collision_data_500m_new.csv'.")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# 1. LOAD DATA
# -------------------------
CSV_PATH = 'enriched_collision_data_500m_new.csv'
df = pd.read_csv(CSV_PATH)

# -------------------------
# 2. EXACT TARGET MAPPING
# -------------------------
# Map Fatal/Serious to 'Serious/Fatal', everything else to 'Slight'
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})
df['Severity_Binary'] = df['Severity_Binary'].apply(lambda x: 'Serious/Fatal' if x == 'Serious/Fatal' else 'Slight')

# LabelEncoder results in: 'Serious/Fatal' -> 0, 'Slight' -> 1
le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])

print(f"Mapping confirmed: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# -------------------------
# 3. FEATURE PREPARATION
# -------------------------

# Filter numeric features and drop non-feature columns
cols_to_drop = ['Severity', 'Severity_Binary', 'dt_combined', 'accident_severity_normalized']
X = df.drop(columns=cols_to_drop, errors='ignore').select_dtypes(include=[np.number])

# Clean data: Remove columns that are all NaN and fill others
X = X.dropna(axis=1, how='all')
X = X.fillna(X.median()).fillna(0)

# -------------------------
# 4. MUTUAL INFORMATION
# -------------------------
print(f"Calculating Mutual Information for {X.shape[1]} features...")
mi_scores = mutual_info_classif(X, y, random_state=42)

# Sort results
mi_results = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# Save to CSV
mi_results.to_csv('top_features_mutual_info.csv')
print("Saved all feature scores to 'top_features_mutual_info.csv'")

# Print top 30 for your Grid Search
print("\n--- Top 30 Features for Grid Search ---")
print(mi_results.head(30).index.tolist())

# We scale the height based on the number of features (e.g., 0.2 inches per feature)
plt.figure(figsize=(12, len(mi_results) * 0.2))
sns.set_style("whitegrid")

# Create plot
sns.barplot(x=mi_results.values, y=mi_results.index, palette='viridis')

plt.title('Complete Mutual Information Scores (All Features)', fontsize=18, fontweight='bold')
plt.xlabel('Mutual Information Score', fontsize=14)
plt.ylabel('Features', fontsize=10)
plt.axvline(x=0, color='black', lw=1) # Baseline

plt.tight_layout()
plt.savefig('all_features_mutual_info.png', dpi=300)
plt.show()

print("\nFull visualization saved as 'all_features_mutual_info.png'.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
CSV_PATH = 'enriched_collision_data_500m_new.csv'
POS_CLASS = 'Serious/Fatal'
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42

# -------------------------
# 1. Load and Prepare
# -------------------------
df = pd.read_csv(CSV_PATH)

# Binary Target
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})
df['Severity_Binary'] = df['Severity_Binary'].apply(lambda x: 'Serious/Fatal' if x == 'Serious/Fatal' else 'Slight')

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)
pos_label_idx = int(np.where(np.array(class_names) == POS_CLASS)[0][0])
other_label_idx = 1 - pos_label_idx

# Candidate numeric features (raw, no preprocessing yet)
X_full_numeric = df.select_dtypes(include=[np.number]).drop(columns=['accident_severity_normalized'], errors='ignore')

manual_top_10 = [
    'collision_year', 'longitude', 'latitude', 'road_type',
    'second_road_class', 'light_conditions', 'weather_conditions',
    'road_surface_conditions', 'urban_or_rural_area', 'trunk_road_flag'
]

# ----------------------------------------------------------------
# STEP 1: MANUAL GRID SEARCH (LEAKAGE-FREE)
# ----------------------------------------------------------------
param_grid = {
    "hidden_layer_sizes": [(8,8), (8,4), (16,8), (8,)],
    "alpha": [1e-4, 1e-3],
    "sampling_strategy": [0.7, 1.0],
    "n_neighbors": [3, 5],
    "threshold": [0.40, 0.45],
}

candidates = list(ParameterGrid(param_grid))
total_candidates = len(candidates)

print(f"Total Candidates: {total_candidates}")
print(f"Executing {N_REPEATS * N_SPLITS} fits per candidate (Total: {total_candidates * (N_REPEATS * N_SPLITS)})")
print("-" * 115)
print(f"{'#':<4} | {'Layers':<12} | {'Alpha':<8} | {'Samp':<5} | {'Neigh':<5} | {'Thresh':<6} | {'Bal. Acc':<10} | {'Std Dev':<9} | {'Status'}")
print("-" * 115)

best_score = -1
best_params_overall = None

cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

# ----------------------------------------------------------------
# MAIN GRID SEARCH LOOP (NO LEAKAGE)
# ----------------------------------------------------------------
for i, params in enumerate(candidates):
    fold_scores = []

    for train_idx, val_idx in cv.split(X_full_numeric, y):

        # Split raw data FIRST
        X_tr_raw = X_full_numeric.iloc[train_idx].copy()
        X_val_raw = X_full_numeric.iloc[val_idx].copy()
        y_tr, y_val = y[train_idx], y[val_idx]

        # -------------------------
        # 1. Imputation (TRAIN ONLY)
        # -------------------------
        medians = X_tr_raw.median()
        X_tr_raw = X_tr_raw.fillna(medians).fillna(0)
        X_val_raw = X_val_raw.fillna(medians).fillna(0)

        # -------------------------
        # 2. Density filtering (TRAIN ONLY)
        # -------------------------
        dense_cols = X_tr_raw.columns[(X_tr_raw > 0).mean() > 0.10]
        X_tr_dense = X_tr_raw[dense_cols]
        X_val_dense = X_val_raw[dense_cols]

        # -------------------------
        # 3. Mutual Information Selection (TRAIN ONLY)
        # -------------------------
        mi_scores = mutual_info_classif(X_tr_dense, y_tr, random_state=RANDOM_STATE)
        mi_results = pd.Series(mi_scores, index=X_tr_dense.columns).sort_values(ascending=False)

        top_seg = mi_results[mi_results.index.str.startswith('seg_')].head(10).index.tolist()
        top_other = mi_results[
            ~mi_results.index.str.startswith('seg_') &
            ~mi_results.index.isin(manual_top_10)
        ].head(5).index.tolist()

        final_features = list(dict.fromkeys(
            [f for f in manual_top_10 if f in X_tr_dense.columns] + top_seg + top_other
        ))

        X_tr = X_tr_dense[final_features]
        X_val = X_val_dense[final_features]

        # -------------------------
        # 4. Scaling (TRAIN ONLY)
        # -------------------------
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        # -------------------------
        # 5. ADASYN + MLP
        # -------------------------
        ada = ADASYN(
            sampling_strategy=params["sampling_strategy"],
            n_neighbors=params["n_neighbors"],
            random_state=RANDOM_STATE
        )
        X_res, y_res = ada.fit_resample(X_tr, y_tr)

        mlp = MLPClassifier(
            hidden_layer_sizes=params["hidden_layer_sizes"],
            alpha=params["alpha"],
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE
        )
        mlp.fit(X_res, y_res)

        # Predict with custom threshold
        probs = mlp.predict_proba(X_val)[:, pos_label_idx]
        preds = np.where(probs > params["threshold"], pos_label_idx, other_label_idx)

        fold_scores.append(balanced_accuracy_score(y_val, preds))

    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores, ddof=1)

    if mean_score > best_score:
        best_score = mean_score
        best_params_overall = params.copy()
        best_params_overall["std"] = std_score

    print(f"{i+1:<4} | {str(params['hidden_layer_sizes']):<12} | {params['alpha']:<8} | "
          f"{params['sampling_strategy']:<5} | {params['n_neighbors']:<5} | {params['threshold']:<6} | "
          f"{mean_score:.4f}     | {std_score:.4f}    | {total_candidates - (i+1)} left")

# ----------------------------------------------------------------
# STEP 2: FINAL EVALUATION (NO LEAKAGE)
# ----------------------------------------------------------------
print("\n" + "="*50)
print("WINNER FOUND")
print(f"Balanced Acc: {best_score:.4f} (+/- {best_params_overall['std']:.4f})")
print(best_params_overall)
print("="*50)

all_y_true, all_y_pred = [], []
aggregate_cm = np.zeros((2, 2), dtype=int)

for train_idx, test_idx in cv.split(X_full_numeric, y):

    X_tr_raw = X_full_numeric.iloc[train_idx].copy()
    X_te_raw = X_full_numeric.iloc[test_idx].copy()
    y_tr, y_te = y[train_idx], y[test_idx]

    # Impute
    medians = X_tr_raw.median()
    X_tr_raw = X_tr_raw.fillna(medians).fillna(0)
    X_te_raw = X_te_raw.fillna(medians).fillna(0)

    # Density filter
    dense_cols = X_tr_raw.columns[(X_tr_raw > 0).mean() > 0.10]
    X_tr_dense = X_tr_raw[dense_cols]
    X_te_dense = X_te_raw[dense_cols]

    # MI selection
    mi_scores = mutual_info_classif(X_tr_dense, y_tr, random_state=RANDOM_STATE)
    mi_results = pd.Series(mi_scores, index=X_tr_dense.columns).sort_values(ascending=False)

    top_seg = mi_results[mi_results.index.str.startswith('seg_')].head(10).index.tolist()
    top_other = mi_results[
        ~mi_results.index.str.startswith('seg_') &
        ~mi_results.index.isin(manual_top_10)
    ].head(5).index.tolist()

    final_features = list(dict.fromkeys(
        [f for f in manual_top_10 if f in X_tr_dense.columns] + top_seg + top_other
    ))

    X_tr = X_tr_dense[final_features]
    X_te = X_te_dense[final_features]

    # Scale
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    # ADASYN + MLP
    ada = ADASYN(
        sampling_strategy=best_params_overall["sampling_strategy"],
        n_neighbors=best_params_overall["n_neighbors"],
        random_state=RANDOM_STATE
    )
    X_res, y_res = ada.fit_resample(X_tr, y_tr)

    mlp = MLPClassifier(
        hidden_layer_sizes=best_params_overall["hidden_layer_sizes"],
        alpha=best_params_overall["alpha"],
        max_iter=500,
        early_stopping=True,
        random_state=RANDOM_STATE
    )
    mlp.fit(X_res, y_res)

    probs = mlp.predict_proba(X_te)[:, pos_label_idx]
    preds = np.where(probs > best_params_overall["threshold"], pos_label_idx, other_label_idx)

    all_y_true.extend(y_te)
    all_y_pred.extend(preds)
    aggregate_cm += confusion_matrix(y_te, preds, labels=[0, 1])

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

# Confusion Matrix Plot
plt.figure(figsize=(7, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Final Model Results\nMean Balanced Acc: {best_score:.4f}")
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.show()

In [ ]:
# now we match based on last 3 years

import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. Load the datasets
collision_df = pd.read_csv('final_merged_cleaned_data_no_duplicates.csv')
mapping_df = pd.read_csv('metadata_images_mapping.csv')
features_df = pd.read_csv('processed_features_conf_0.7.csv')

# Clean missing values
mapping_df = mapping_df.dropna(subset=['image_blob_name', 'lat', 'lng'])
features_df = features_df.dropna(subset=['image_name'])

# --- TEMPORAL DATA PREP ---
# Ensure collision_year is numeric
if 'collision_year' in collision_df.columns:
    collision_df['collision_year'] = pd.to_numeric(collision_df['collision_year'], errors='coerce')

# Extract year from mapping_df metadata if not already present
if 'year' not in mapping_df.columns:
    if 'date' in mapping_df.columns:
        mapping_df['year'] = pd.to_datetime(mapping_df['date'], errors='coerce').dt.year
    else:
        mapping_df['year'] = np.nan
mapping_df['year'] = pd.to_numeric(mapping_df['year'], errors='coerce')

# 2. Link Features to Coordinates
mapping_df['image_filename'] = mapping_df['image_blob_name'].astype(str).apply(lambda x: x.split('/')[-1])

# Merge visual features with their lat/lng and year
visual_data = pd.merge(
    features_df,
    mapping_df[['image_filename', 'lat', 'lng', 'year']],
    left_on='image_name',
    right_on='image_filename',
    how='inner'
)

# Average the features per unique location AND year
# This ensures we handle locations that were photographed in multiple different years
visual_features_per_loc = visual_data.drop(columns=['image_name', 'image_filename']).groupby(['lat', 'lng', 'year']).mean().reset_index()

# 3. Spatial Search (100m Radius)
mean_lat = np.radians(51.5)  # Approximate latitude for London
lng_scale = np.cos(mean_lat)

visual_coords = visual_features_per_loc[['lat', 'lng']].values
visual_coords_scaled = visual_coords.copy()
visual_coords_scaled[:, 1] *= lng_scale

collision_coords = collision_df[['latitude', 'longitude']].values
collision_coords_scaled = collision_coords.copy()
collision_coords_scaled[:, 1] *= lng_scale

# Build spatial index
tree = cKDTree(visual_coords_scaled)

# 100 meters expressed in degrees latitude
radius_deg = 100 / 111320.0

# --- UPDATED MATCHING LOGIC (Spatial + Temporal) ---
# Query for up to 10 nearest neighbors to find one that satisfies the year constraint
distances, indices = tree.query(collision_coords_scaled, k=10, distance_upper_bound=radius_deg)

# 4. Merge Data
feature_cols = visual_features_per_loc.columns.drop(['lat', 'lng', 'year'])
matched_feature_values = np.full((len(collision_df), len(feature_cols)), np.nan)
matched_mask = np.zeros(len(collision_df), dtype=bool)

# Pre-extract values for fast processing
v_years = visual_features_per_loc['year'].values
v_feats = visual_features_per_loc[feature_cols].values
c_years = collision_df['collision_year'].values if 'collision_year' in collision_df.columns else np.full(len(collision_df), np.nan)

print("Applying spatial and temporal filters (3-year window)...")
for i in range(len(collision_df)):
    c_year = c_years[i]
    if np.isnan(c_year):
        continue

    for k_idx in range(10):
        v_idx = indices[i, k_idx]

        # Check if neighbor exists within distance radius (cKDTree returns len(data) if no match)
        if v_idx >= len(visual_features_per_loc):
            break

        v_year = v_years[v_idx]
        if np.isnan(v_year):
            continue

        # Apply the 5-year temporal constraint
        if abs(c_year - v_year) <= 3:
            matched_feature_values[i] = v_feats[v_idx]
            matched_mask[i] = True
            break # Take the spatially closest image that fits the time window

# Create final DataFrame (enriched)
features_to_add = pd.DataFrame(matched_feature_values, columns=feature_cols, index=collision_df.index)
enriched_df = pd.concat([collision_df, features_to_add], axis=1)

total_collisions = len(collision_df)
matched_count = int(matched_mask.sum())
matched_pct = matched_mask.mean() * 100.0

print(f"Total collisions (input): {total_collisions}")
print(f"Matched with features (spatial + temporal): {matched_count} ({matched_pct:.2f}%)")

# Only keep matched rows for final CSV
if matched_count == 0:
    print("No collisions were matched within 100m and 3 years. No output CSV written.")
else:
    matched_df = enriched_df.loc[matched_mask].copy()

    # Automatically identify severity column
    severity_candidates = ['accident_severity', 'severity', 'Accident_Severity', 'severity_label', 'acc_sev']
    severity_col = next((c for c in severity_candidates if c in matched_df.columns), None)

    if severity_col is None:
        for c in matched_df.columns:
            if 'sever' in c.lower() or 'accident' in c.lower():
                severity_col = c
                break

    if severity_col is None:
        matched_df.to_csv('enriched_collision_data_100m_matched.csv', index=False)
        print("Saved data (no severity stats).")
    else:
        print(f"Detected severity column: '{severity_col}'")

        def normalize_severity(s):
            if pd.isna(s): return np.nan
            if isinstance(s, (int, float)):
                if int(s) == 1: return 'Fatal'
                if int(s) == 2: return 'Serious'
                if int(s) == 3: return 'Slight'
            s_str = str(s).strip().lower()
            if 'fatal' in s_str or s_str == '1': return 'Fatal'
            if 'serious' in s_str or s_str == '2': return 'Serious'
            if 'slight' in s_str or 'minor' in s_str or s_str == '3': return 'Slight'
            return s_str

        matched_df['accident_severity_normalized'] = matched_df[severity_col].map(normalize_severity)
        counts = matched_df['accident_severity_normalized'].value_counts()

        print("\nMatched set severity breakdown:")
        for cat in ['Fatal', 'Serious', 'Slight']:
            cnt = int(counts.get(cat, 0))
            p = (cnt / matched_count * 100.0) if matched_count > 0 else 0
            print(f"  {cat}: {cnt} ({p:.2f}%)")

        matched_df.to_csv('enriched_collision_data_100m_matched.csv', index=False)
        print("\nSaved matched data to 'enriched_collision_data_100m_matched.csv'.")

In [ ]:
import pandas as pd
import numpy as np

def apply_mappings(df, year):
    """
    Applies the specific value mappings and logic provided in the user's table.
    """
    # 1. Handle 2024 Header Truncations & Year-Specific Renaming
    if year == 2024:
        rename_map = {
            'collisio_1': 'collision_year',
            'collisio_2': 'collision_ref_no',
            'collisio_3': 'Severity',
            'day_of_wee': 'day_of_week',
            'first_road': 'first_road_class',
            'speed_limi': 'speed_limit',
            'second_roa': 'second_road_class',
            'urban_or_r': 'urban_or_rural_area',
            'trunk_road': 'trunk_road_flag',
            'weather_co': 'weather_conditions',
            'road_surfa': 'road_surface_conditions',
            'special_co': 'special_conditions_at_site',
            'light_cond': 'light_conditions',
            'junction_d': 'junction_detail_historic',
            'junction_1': 'junction_detail',
            'pedestri_2': 'pedestrian_crossing',
            'pedestri_1': 'pedestrian_crossing_physical_facilities_historic',
            'pedestrian': 'pedestrian_crossing_human_control_historic',
            'carriagewa': 'carriageway_hazards_historic',
            'carriage_1': 'carriageway_hazards'
        }
        df = df.rename(columns=rename_map)
    else:
        # Standardize Severity column name for 2013-2023
        if 'Severity' not in df.columns:
            if 'collision_severity' in df.columns:
                df['Severity'] = df['collision_severity']

    # Ensure all columns required for logic exist to avoid errors
    logic_cols = [
        'junction_detail', 'junction_detail_historic',
        'pedestrian_crossing', 'pedestrian_crossing_human_control_historic',
        'pedestrian_crossing_physical_facilities_historic',
        'carriageway_hazards', 'carriageway_hazards_historic'
    ]
    for col in logic_cols:
        if col not in df.columns:
            df[col] = np.nan

    # 2. Junction Detail Mapping Logic (Historic -> Modern)
    jd_map = {3.0: 13.0, 6.0: 16.0, 7.0: 17.0, 8.0: 18.0, 9.0: 19.0, 99.0: -1.0}
    mask_jd = (df['junction_detail'].isna()) | (df['junction_detail'].isin([0, 1, 2, 3, 5, 6, 7, 8, 9]))
    df.loc[mask_jd, 'junction_detail'] = df.loc[mask_jd, 'junction_detail_historic'].map(jd_map).fillna(df.loc[mask_jd, 'junction_detail'])

    # 3. Pedestrian Crossing Mapping Logic (Human + Physical -> Combined)
    h_map = {1.0: 11.0, 2.0: 12.0}
    p_map = {1.0: 13.0, 4.0: 14.0, 5.0: 15.0, 7.0: 16.0, 8.0: 17.0}

    mask_h = df['pedestrian_crossing_human_control_historic'].isin(h_map.keys())
    df.loc[mask_h, 'pedestrian_crossing'] = df.loc[mask_h, 'pedestrian_crossing_human_control_historic'].map(h_map)

    mask_p = df['pedestrian_crossing_physical_facilities_historic'].isin(p_map.keys())
    df.loc[mask_p, 'pedestrian_crossing'] = df.loc[mask_p, 'pedestrian_crossing_physical_facilities_historic'].map(p_map)

    # 4. Carriageway Hazards Mapping Logic (Historic -> Modern)
    haz_map = {1: 16, 2: 17, 3: 18, 6: 20, 7: 19, 9: -1}
    mask_haz = (df['carriageway_hazards'].isna()) | (df['carriageway_hazards'] == 0)
    df.loc[mask_haz, 'carriageway_hazards'] = df.loc[mask_haz, 'carriageway_hazards_historic'].map(haz_map).fillna(df.loc[mask_haz, 'carriageway_hazards'])

    # 5. Severity Label Standardization
    sev_map = {
        '1': 'Fatal', '1.0': 'Fatal', 'Fatal': 'Fatal',
        '2': 'Serious', '2.0': 'Serious', 'Serious': 'Serious',
        '3': 'Slight', '3.0': 'Slight', 'Slight': 'Slight'
    }
    df['Severity'] = df['Severity'].astype(str).str.strip().str.title().map(sev_map)
    # Extract Month from 'date' (assumes DD/MM/YYYY)
    if 'date' in df.columns:
        df['collision_month'] = df['date'].astype(str).str.split('/').str[1]

    # Extract Hour from 'time' (assumes HH:MM:SS)
    if 'time' in df.columns:
        df['collision_hour'] = df['time'].astype(str).str.split(':').str[0]

    return df

# Columns to keep for the final file based on your "Kept" list
final_kept_cols = [
    'collision_year', 'longitude', 'latitude', 'first_road_class', 'road_type',
    'speed_limit', 'second_road_class', 'road_surface_conditions',
    'special_conditions_at_site', 'urban_or_rural_area', 'trunk_road_flag',
    'junction_detail', 'carriageway_hazards', 'pedestrian_crossing',
    'weather_conditions', 'light_conditions', 'day_of_week', 'Severity', 'collision_month', 'collision_hour', 'date', 'time',
]

years = list(range(2013, 2025))
all_data = []

for year in years:
    file_path = f"{year}.csv"
    try:
        df = pd.read_csv(file_path, low_memory=False)

        # Deduplication: identify unique ID per year
        id_col = 'collisio_2' if year == 2024 else 'collision_ref_no'
        df = df.drop_duplicates(subset=[id_col])

        # Apply logic mappings
        df = apply_mappings(df, year)

        # Standardize columns
        for col in final_kept_cols:
            if col not in df.columns:
                df[col] = np.nan

        all_data.append(df[final_kept_cols])
        print(f"Successfully processed {year}")

    except FileNotFoundError:
        print(f"File {file_path} not found.")

# Combine all years into one DataFrame
final_df = pd.concat(all_data, ignore_index=True)

# Final Clean: Year to integer
final_df['collision_year'] = final_df['collision_year'].astype(int)

# Export the file
final_df.to_csv('final_mapped_accidents_2013_2024.csv', index=False)

In [ ]:
# Chatgpt suggestion

import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.metrics import balanced_accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# -----------------------
# 1) Load
# -----------------------
df = pd.read_csv("enriched_collision_data_500m_filtered_relevant.csv")

# -----------------------
# 2) Sanitize column names
# -----------------------
def sanitize(col: str) -> str:
    col = re.sub(r"[()]", "", col)
    col = col.replace(" ", "_")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = re.sub(r"__+", "_", col)
    return col

df = df.rename(columns={c: sanitize(c) for c in df.columns})

# -----------------------
# 3) Target encoding
# -----------------------
y_map = {"Slight": 0, "Serious": 1, "Fatal": 1}
df["y"] = df["Severity"].map(y_map).astype(int)

# -----------------------
# 4) Feature engineering (time)
# -----------------------
date = pd.to_datetime(dict(year=df["year"], month=df["month"], day=df["day"]), errors="coerce")
df["day_of_year"] = date.dt.dayofyear.astype(int)
df["time_of_day"] = df["hour"] * 60 + df["minutes"]

df["day_of_year_sin"] = np.sin(2*np.pi*df["day_of_year"]/365.25)
df["day_of_year_cos"] = np.cos(2*np.pi*df["day_of_year"]/365.25)
df["time_of_day_sin"] = np.sin(2*np.pi*df["time_of_day"]/1440.0)
df["time_of_day_cos"] = np.cos(2*np.pi*df["time_of_day"]/1440.0)

# In your data, day_of_week matches: 1=Sunday, ..., 7=Saturday
df["is_weekend"] = df["day_of_week"].isin([1, 7]).astype(int)

# Drop raw redundant time columns after encoding (optional but recommended)
drop_time_raw = ["month", "day", "hour", "minutes", "day_of_year", "time_of_day"]

# -----------------------
# 5) Identify columns
# -----------------------
seg_cols = [c for c in df.columns if c.startswith("seg_")]
obj_cols = [c for c in df.columns if c.startswith("obj_")]

cat_cols = [
    "first_road_class", "road_type", "second_road_class",
    "road_surface_conditions", "special_conditions_at_site",
    "urban_or_rural_area", "trunk_road_flag", "junction_detail",
    "carriageway_hazards", "pedestrian_crossing",
    "weather_conditions", "light_conditions", "day_of_week"
]

num_cols = (
    ["longitude", "latitude", "speed_limit", "year",
     "day_of_year_sin", "day_of_year_cos", "time_of_day_sin", "time_of_day_cos",
     "is_weekend"]
    + seg_cols + obj_cols
)

# -----------------------
# 6) Clean sentinel values
# -----------------------
# speed_limit: -1 = unknown -> treat as missing
df["speed_limit"] = df["speed_limit"].replace(-1, np.nan)

# categorical unknown mapping (adapt if your codebook differs)
for c in cat_cols:
    if c in {"second_road_class", "junction_detail", "carriageway_hazards",
             "road_surface_conditions", "special_conditions_at_site"}:
        df[c] = df[c].replace(-1, "UNK")
    if c in {"road_type", "road_surface_conditions", "special_conditions_at_site", "weather_conditions"}:
        df[c] = df[c].replace(9, "UNK")
    if c in {"carriageway_hazards", "pedestrian_crossing"}:
        df[c] = df[c].replace(99, "UNK")
    df[c] = df[c].astype(str)

# log1p object features
for c in obj_cols:
    df[c] = np.log1p(df[c])

# -----------------------
# 7) Temporal split (recommended)
# -----------------------
train_df = df[df["year"] <= 2022].copy()
val_df   = df[df["year"] == 2023].copy()
test_df  = df[df["year"] == 2024].copy()

X_train = train_df[cat_cols + num_cols].drop(columns=[c for c in drop_time_raw if c in train_df.columns], errors="ignore")
y_train = train_df["y"].values

X_val = val_df[cat_cols + num_cols].drop(columns=[c for c in drop_time_raw if c in val_df.columns], errors="ignore")
y_val = val_df["y"].values

X_test = test_df[cat_cols + num_cols].drop(columns=[c for c in drop_time_raw if c in test_df.columns], errors="ignore")
y_test = test_df["y"].values

# -----------------------
# 8) Preprocess: scale numeric + one-hot categorical
# -----------------------
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

X_train_p = preprocess.fit_transform(X_train)
X_val_p   = preprocess.transform(X_val)
X_test_p  = preprocess.transform(X_test)

# Convert to dense (small enough after one-hot). If you have memory issues, keep sparse and use a linear model baseline.
X_train_np = X_train_p.toarray() if hasattr(X_train_p, "toarray") else X_train_p
X_val_np   = X_val_p.toarray()   if hasattr(X_val_p, "toarray") else X_val_p
X_test_np  = X_test_p.toarray()  if hasattr(X_test_p, "toarray") else X_test_p

# -----------------------
# 9) Class weights (cap Fatal weight to avoid instability)
# -----------------------
classes = np.array([0, 1])
w = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)

# cap the rare-class weight (tune this; 30–50 often safer than 200+)
w[1] = min(w[1], 50.0)

# -----------------------
# 10) PyTorch model
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

Xtr = torch.tensor(X_train_np, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xva = torch.tensor(X_val_np, dtype=torch.float32)
yva = torch.tensor(y_val, dtype=torch.long)
Xte = torch.tensor(X_test_np, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, yva), batch_size=512, shuffle=False)

class MLP(nn.Module):
    def __init__(self, in_dim, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(Xtr.shape[1]).to(device)

criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def eval_macro_f1(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds.append(torch.argmax(logits, dim=1).cpu().numpy())
            trues.append(yb.cpu().numpy())
    y_true = np.concatenate(trues)
    y_pred = np.concatenate(preds)
    return f1_score(y_true, y_pred, average="macro"), y_true, y_pred

best_f1 = -1
patience = 10
pat = 0

for epoch in range(200):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    val_f1, _, _ = eval_macro_f1(val_loader)
    print(f"Epoch {epoch:03d} | val macro-F1={val_f1:.4f}")

    if val_f1 > best_f1 + 1e-4:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print("Early stopping.")
            break

# restore best
model.load_state_dict(best_state)

# -----------------------
# 11) Final test evaluation
# -----------------------
model.eval()
with torch.no_grad():
    logits = model(Xte.to(device)).cpu().numpy()
y_pred = logits.argmax(axis=1)

# Confusion matrix + classification report
print(confusion_matrix(y_test, y_pred))
print(classification_report(
    y_test, y_pred,
    target_names=["Slight", "Serious_or_Fatal"]
))

# ---- Balanced Accuracy ----
bal_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced Accuracy: {bal_acc:.4f}")

# ---- Std deviation of balanced accuracy (bootstrap) ----
n_boot = 1000
boot_scores = []

rng = np.random.default_rng(42)
for _ in range(n_boot):
    idx = rng.integers(0, len(y_test), len(y_test))
    boot_scores.append(
        balanced_accuracy_score(y_test[idx], y_pred[idx])
    )

std_bal_acc = np.std(boot_scores)
print(f"Balanced Accuracy Std (bootstrap): {std_bal_acc:.4f}")

In [ ]:


import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ---- Helpers ----
def latlon_to_ecef(lat_deg, lon_deg, h=0.0):
    """
    Convert latitude/longitude (degrees) to ECEF Cartesian coordinates (meters)
    using the WGS84 ellipsoid.
    Returns array shape (N,3).
    """
    lat = np.radians(np.asarray(lat_deg, dtype=float))
    lon = np.radians(np.asarray(lon_deg, dtype=float))
    a = 6378137.0  # semi-major axis (m)
    f = 1.0 / 298.257223563  # flattening
    e2 = f * (2 - f)

    sin_lat = np.sin(lat)
    cos_lat = np.cos(lat)
    sin_lon = np.sin(lon)
    cos_lon = np.cos(lon)

    N = a / np.sqrt(1.0 - e2 * sin_lat * sin_lat)
    x = (N + h) * cos_lat * cos_lon
    y = (N + h) * cos_lat * sin_lon
    z = ((1.0 - e2) * N + h) * sin_lat

    return np.column_stack((x, y, z))


# ---- 1. Load the datasets ----
collision_df = pd.read_csv('final_mapped_accidents_with_distances.csv')
collision_df = collision_df.drop(columns=['year'])
mapping_df = pd.read_csv('metadata_images_mapping.csv')
features_df = pd.read_csv('processed_features_conf_0.5_probs.csv')

# Clean missing values
mapping_df = mapping_df.dropna(subset=['image_blob_name', 'lat', 'lng'])
features_df = features_df.dropna(subset=['image_name'])
features_df = features_df.fillna(0)
# ---- 2. Link Features to Coordinates ----
# Extract the base filename (e.g., '4411_0.jpg') from the full blob path
mapping_df['image_filename'] = mapping_df['image_blob_name'].astype(str).apply(lambda x: x.split('/')[-1])

# Merge visual features with their lat/lng coordinates
visual_data = pd.merge(
    features_df,
    mapping_df[['image_filename', 'lat', 'lng', 'year']],
    left_on='image_name',
    right_on='image_filename',
    how='inner'
)

visual_features_per_loc = (
    visual_data.drop(columns=['image_name', 'image_filename'])
    .groupby(['lat', 'lng'])
    .mean()
    .reset_index()
)

visual_features_per_loc['infrastructure_lat'] = visual_features_per_loc['lat']
visual_features_per_loc['infrastructure_long'] = visual_features_per_loc['lng']
# === END ADDITION ===
# ---- 3. Convert lat/lon to ECEF Cartesian coordinates and build KD-tree ----
# Visual locations (lat, lng)
vis_lats = visual_features_per_loc['lat'].values
vis_lons = visual_features_per_loc['lng'].values
visual_ecef = latlon_to_ecef(vis_lats, vis_lons)  # in meters

# Collision locations (latitude, longitude) - adapt column names if different
if not {'latitude', 'longitude'}.issubset(collision_df.columns):
    raise KeyError("collision_df must contain 'latitude' and 'longitude' columns")

col_lats = collision_df['latitude'].values
col_lons = collision_df['longitude'].values
collision_ecef = latlon_to_ecef(col_lats, col_lons)  # in meters

# Build spatial index on Cartesian coordinates (meters)
tree = cKDTree(visual_ecef)

# Find the nearest visual feature for every accident (no distance upper bound)
# distances returned are Euclidean meters (since ECEF is in meters)
distances_m, indices = tree.query(collision_ecef, k=1)

# ---- 4. Merge Data (match every collision to its nearest visual feature) ----
# Prepare feature columns from visual_features_per_loc
feature_cols = visual_features_per_loc.columns.drop(['lat', 'lng'])

# Create DataFrame of matched visual features for each collision using the indices
# Note: indices is an array of integers with length == len(collision_df)
matched_visual_rows = visual_features_per_loc.iloc[indices][feature_cols].reset_index(drop=True)

# Prepare final enriched dataframe by concatenating matched features and the distance column
features_to_add = matched_visual_rows.set_index(collision_df.index)
enriched_df = pd.concat([collision_df.reset_index(drop=True), features_to_add], axis=1)
enriched_df['distance_from_point'] = distances_m  # meters


# ---- 5. Reporting ----
total_collisions = len(collision_df)
matched_count = total_collisions  # we matched every collision to nearest visual point
matched_pct = 100.0

print(f"Total collisions (input): {total_collisions}")
print(f"Matched with features: {matched_count} ({matched_pct:.2f}%)")
print("Distance units: meters (ECEF Euclidean distance).")

# ---- 6. Severity detection + statistics (same logic as original) ----
matched_df = enriched_df.copy()  # all rows are matched

# --- Identify the severity column automatically (common names) ---
severity_candidates = [
    'accident_severity', 'severity', 'Accident_Severity', 'severity_label',
    'acc_sev', 'severity_code', 'severity_type'
]
severity_col = None
# prefer exact matches first
for c in severity_candidates:
    if c in matched_df.columns:
        severity_col = c
        break

# fallback: look for a column containing 'sever' or 'accident' in its name
if severity_col is None:
    for c in matched_df.columns:
        if 'sever' in c.lower() or 'accident' in c.lower():
            severity_col = c
            break

# If still None, we won't compute severity stats (but will save CSV)
if severity_col is None:
    print("Warning: Could not automatically find a severity column in the matched data.")
    print("Available columns (sample):", list(matched_df.columns[:30]))
    matched_df.to_csv('enriched_collision_data_with_distance_no_severity.csv', index=False)
    print("Saved matched data to 'enriched_collision_data_with_distance_no_severity.csv'.")
else:
    print(f"Detected severity column: '{severity_col}'")

    # Normalize / map severity to canonical labels: 'Fatal', 'Serious', 'Slight'
    sev_series = matched_df[severity_col]

    def normalize_severity(s):
        if pd.isna(s):
            return np.nan
        if isinstance(s, (int, np.integer, float, np.floating)):
            try:
                si = int(s)
                if si == 1:
                    return 'Fatal'
                elif si == 2:
                    return 'Serious'
                elif si == 3:
                    return 'Slight'
            except Exception:
                pass
        s_str = str(s).strip().lower()
        if 'fatal' in s_str or s_str == '1':
            return 'Fatal'
        if 'serious' in s_str or 'serious-injury' in s_str or s_str == '2':
            return 'Serious'
        if 'slight' in s_str or 'minor' in s_str or s_str == '3':
            return 'Slight'
        return s_str

    matched_df['accident_severity_normalized'] = sev_series.map(normalize_severity)

    # Count canonical categories (ensure the requested order)
    categories = ['Fatal', 'Serious', 'Slight']
    counts = matched_df['accident_severity_normalized'].value_counts(dropna=False)

    fatal_count = int(counts.get('Fatal', 0))
    serious_count = int(counts.get('Serious', 0))
    slight_count = int(counts.get('Slight', 0))

    def pct(n):
        return (n / matched_count * 100.0) if matched_count > 0 else 0.0

    fatal_pct = pct(fatal_count)
    serious_pct = pct(serious_count)
    slight_pct = pct(slight_count)

    print("\nMatched set severity breakdown (counts and % of matched set):")
    print(f"  Fatal:   {fatal_count} ({fatal_pct:.2f}%)")
    print(f"  Serious: {serious_count} ({serious_pct:.2f}%)")
    print(f"  Slight:  {slight_count} ({slight_pct:.2f}%)")

    other_labels = [lab for lab in counts.index if lab not in categories and pd.notna(lab)]
    if other_labels:
        print("\nOther / unrecognized severity labels present in matched data (counts):")
        for lab in other_labels:
            print(f"  {lab}: {int(counts[lab])}")
    cols = [c for c in matched_df.columns if c.startswith('obj_') or c.startswith('seg_')]
    if cols:
        matched_df.loc[:, cols] = matched_df.loc[:, cols].fillna(0)
    matched_df = matched_df.drop(columns=['accident_severity_normalized'], errors='ignore')
    if severity_col in matched_df.columns:
      cols = [c for c in matched_df.columns if c != severity_col] + [severity_col]
      matched_df = matched_df[cols]

    obj_prob_cols = sorted(
        [c for c in matched_df.columns if c.startswith('obj_') and not c.endswith('_count')]
    )

    new_order = []
    used = set()

    for col in matched_df.columns:
        if col in used:
            continue

        if col in obj_prob_cols:
            count_col = f"{col}_count"
            new_order.append(col)
            used.add(col)

            if count_col in matched_df.columns:
                new_order.append(count_col)
                used.add(count_col)
        else:
            if col not in used:
                new_order.append(col)
                used.add(col)

    matched_df = matched_df[new_order]
    matched_df = matched_df.rename(columns={'year': 'infrastructure_year'})
    matched_df['infrastructure_year'] = matched_df['infrastructure_year'].astype(int)
    # Save matched results (includes distance_from_point in meters)
    matched_df.to_csv('enriched_collision_data_with_distance.csv', index=False)
    print("\nSaved matched data to 'enriched_collision_data_with_distance.csv'.")

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree

def haversine(lat1, lon1, lat2, lon2):
    """Calculate the great circle distance between two points in meters."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371000 * c

# 1. Load the datasets
df_accidents = pd.read_csv('final_mapped_accidents.csv')
df_distances = pd.read_csv('distances.csv')

# Columns to extract from the distances file
cols_to_merge = [
    'nearest_pub_road_m', 'nearest_pub_crow_m',
    'nearest_school_road_m', 'nearest_school_crow_m',
    'nearest_overground_road_m', 'nearest_overground_crow_m',
    'nearest_underground_road_m', 'nearest_underground_crow_m',
    'nearest_signal_road_m', 'nearest_signal_crow_m'
]

# 2. Prepare Reference Data (Remove duplicates in distance file to ensure 1-to-1 matching)
ref_df = df_distances[['latitude', 'longitude'] + cols_to_merge].drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)
coords_ref = ref_df[['latitude', 'longitude']].values

# 3. Build the KDTree for fast spatial lookup
tree = KDTree(coords_ref)

# 4. Query the tree for every accident coordinate (Nearest Neighbor)
coords_query = df_accidents[['latitude', 'longitude']].values
distances_euclidean, indices = tree.query(coords_query)

# 5. Extract matched columns and the actual coordinates they matched to
matched_data = ref_df.iloc[indices][cols_to_merge].reset_index(drop=True)
matched_coords = ref_df.iloc[indices][['latitude', 'longitude']].reset_index(drop=True)

# 6. Merge columns into the accidents dataframe
final_df = pd.concat([df_accidents, matched_data], axis=1)

# 7. Calculate real-world physical distance (Haversine) for matching accuracy
match_physical_distances = haversine(
    df_accidents['latitude'], df_accidents['longitude'],
    matched_coords['latitude'], matched_coords['longitude']
)

# 8. Calculate Percentages for Report
total_count = len(df_accidents)
within_5m = (match_physical_distances <= 5).sum()
within_10m = (match_physical_distances <= 10).sum()
pct_5m = (within_5m / total_count) * 100
pct_10m = (within_10m / total_count) * 100
max_dist = match_physical_distances.max()

# 9. Print Statistics
print(f"--- Global Matching Statistics ---")
print(f"Total Rows Processed: {total_count}")
print(f"Matched Rows: {len(final_df)} (100.00%)")
print(f"\n--- Distance to Match Stats ---")
print(f"Min Distance: {match_physical_distances.min():.4f} m")
print(f"Max Distance: {max_dist:.4f} m")
print(f"Avg Distance: {match_physical_distances.mean():.4f} m")
print(f"\n{pct_5m:.1f}% of the records matched within 5 meters.")
print(f"{pct_10m:.1f}% of the records matched within 10 meters.")
print(f"The {max_dist:.0f}m result is a single outlier, likely due to a coordinate in the accidents file that is far from any point in the distances database.")

# 10. Save result
final_df.to_csv('final_mapped_accidents_with_distances.csv', index=False)

In [ ]:
# Full Synchronous Open-Meteo Fetcher with 429 Protection
!pip install -q requests pandas python-dateutil tqdm

import os, json, time, csv
from datetime import datetime, timezone
from dateutil import parser
import requests
import pandas as pd
from tqdm import tqdm
import bisect

# === CONFIG ===
INPUT_CSV = "enriched_collision_data_with_distance.csv"
OUTPUT_CSV = "weather_data.csv"
ERRORS_CSV = "errors_log.csv"
RAW_RESP_DIR = "responses_raw"
REQUEST_TIMEOUT = 25
SLEEP_BETWEEN_REQUESTS = 0.0    # Gap between successful requests
MAX_RETRIES = 5                 # More retries for rate limit recovery
BACKOFF_FACTOR = 2.0

HOURLY_VARIABLES_OLD = [
    "temperature_2m", "relativehumidity_2m", "dewpoint_2m", "apparent_temperature",
    "precipitation", "rain", "showers", "snowfall", "snow_depth",
    "weathercode",
    "pressure_msl", "surface_pressure",
    "cloudcover", "cloudcover_low", "cloudcover_mid", "cloudcover_high",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation", "dni", "ghi", "gt", "gti",
    "windspeed_10m", "winddirection_10m", "windgusts_10m",
    "visibility",
    "evapotranspiration", "et0", "vapour_pressure_deficit",
    "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", "soil_temperature_54cm",
    "soil_moisture_0_1cm", "soil_moisture_1_3cm", "soil_moisture_3_9cm", "soil_moisture_9_27cm",
    "cape", "lifted_index", "is_day",
]

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "cloudcover",
    "windspeed_10m",
    "visibility",
]
os.makedirs(RAW_RESP_DIR, exist_ok=True)

# ---------------- HELPERS (Original Logic Preserved) ----------------

def filter_hourly_vars_for_endpoint(hourly_vars, endpoint):
    if "archive-api" in endpoint:
        allowed = {
            "temperature_2m", "relativehumidity_2m", "dewpoint_2m", "apparent_temperature",
            "precipitation", "rain", "showers", "snowfall", "snow_depth",
            "weathercode", "pressure_msl", "surface_pressure",
            "cloudcover", "cloudcover_low", "cloudcover_mid", "cloudcover_high",
            "windspeed_10m", "winddirection_10m", "windgusts_10m", "visibility"
        }
        return [v for v in hourly_vars if v in allowed]
    return hourly_vars

def _parse_datetime_from_row(row):
    for col in ("datetime", "timestamp", "dt"):
        if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
            try:
                dt = parser.isoparse(str(row[col]))
                if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
                return dt.astimezone(timezone.utc)
            except: pass

    def _get_int(cols_list, default=None):
        for c in cols_list:
            if c in row and pd.notna(row[c]) and str(row[c]).strip() != "":
                try: return int(float(str(row[c]).strip()))
                except: return default
        return default

    year, month, day = _get_int(["collision_year", "year"]), _get_int(["month"]), _get_int(["day"])
    if year is None or month is None or day is None: return None
    try:
        return datetime(year, month, day, _get_int(["hour"], 0), _get_int(["minutes","minute"], 0), 0, tzinfo=timezone.utc)
    except: return None

def _choose_endpoint_for_date(dt_utc):
    today_utc = datetime.now(timezone.utc).date()
    return "https://archive-api.open-meteo.com/v1/archive" if dt_utc.date() < today_utc else "https://api.open-meteo.com/v1/forecast"

def _request_open_meteo_once(lat, lon, dt_utc, hourly_vars):
    date_str = dt_utc.date().isoformat()
    endpoint = _choose_endpoint_for_date(dt_utc)
    filtered_vars = filter_hourly_vars_for_endpoint(hourly_vars, endpoint)
    params = {
        "latitude": float(lat), "longitude": float(lon),
        "start_date": date_str, "end_date": date_str,
        "hourly": ",".join(filtered_vars), "timezone": "UTC",
    }
    if endpoint.endswith("/forecast"): params["current_weather"] = "true"

    resp = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
    return resp

def request_with_retries(lat, lon, dt_utc, hourly_vars):
    attempt = 0
    wait = 2.0
    while attempt < MAX_RETRIES:
        try:
            resp = _request_open_meteo_once(lat, lon, dt_utc, hourly_vars)

            if resp.status_code == 429:
                # Specific handling for Rate Limit
                pause_time = 60 if attempt < 2 else 300 # Wait 1 min, then 5 mins if it persists
                print(f"\n[!] Rate Limited (429). Sleeping {pause_time}s before retry {attempt+1}...")
                time.sleep(pause_time)
                attempt += 1
                continue

            resp.raise_for_status()
            return True, (resp.json(), resp.url)

        except Exception as e:
            print(f"\n[!] Connection error on row: {e}. Retrying in {wait}s...")
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, "Max retries exceeded or persistent 429"

def _find_best_hour_index(times_list, target_dt_utc):
    target_iso = target_dt_utc.strftime("%Y-%m-%dT%H:%M")
    try:
        idx = times_list.index(target_iso)
        return idx, "exact", times_list[idx]
    except ValueError:
        pos = bisect.bisect_left(times_list, target_iso)
        candidates = [p for p in [pos - 1, pos] if 0 <= p < len(times_list)]
        def iso_to_dt(s): return parser.isoparse(s).replace(tzinfo=timezone.utc)
        best = min(candidates, key=lambda i: abs((iso_to_dt(times_list[i]) - target_dt_utc).total_seconds()))
        return best, "nearest", times_list[best]

# ---------------- WORKER (Called Synchronously) ----------------

def _process_row_sync(idx, row):
    try:
        row = row.fillna("")
        lat, lon = row.get("latitude"), row.get("longitude")
        if not lat or not lon: return (idx, False, "missing lat/lon", None)

        dt_utc = _parse_datetime_from_row(row)
        if dt_utc is None: return (idx, False, "missing or unparseable datetime", None)

        ok, payload_or_err = request_with_retries(lat, lon, dt_utc, HOURLY_VARIABLES)
        if not ok: return (idx, False, f"request failed: {payload_or_err}", None)

        payload, url = payload_or_err

        # Raw saving
        try:
            with open(os.path.join(RAW_RESP_DIR, f"row_{idx}_raw.json"), "w") as fh:
                json.dump(payload, fh)
        except: pass

        hourly = payload.get("hourly", {})
        times = hourly.get("time")
        if not times: return (idx, False, "no hourly.time in response", url)

        idx_hour, match_type, returned_time = _find_best_hour_index(times, dt_utc)

        result_row = {col: row[col] for col in row.index}
        result_row.update({
            "_row_index": idx, "requested_datetime_utc": dt_utc.isoformat(),
            "returned_time_utc": returned_time, "match_type": match_type
        })

        for meta in ("latitude", "longitude", "generationtime_ms", "utc_offset_seconds"):
            if meta in payload: result_row[meta] = payload[meta]

        for var in HOURLY_VARIABLES:
            series = hourly.get(var)
            result_row[var] = series[idx_hour] if (series and idx_hour < len(series)) else None

        return (idx, True, result_row, url)
    except Exception as e:
        return (idx, False, f"unexpected error: {e}", None)

# ---------------- MAIN STREAMING LOOP ----------------

def run_fetcher():
    df = pd.read_csv(INPUT_CSV, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    if os.path.exists(OUTPUT_CSV): os.remove(OUTPUT_CSV)
    if os.path.exists(ERRORS_CSV): os.remove(ERRORS_CSV)

    header_written = False
    success_count, error_count = 0, 0

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Fetching Rows"):
        idx, ok, payload, url = _process_row_sync(i, row)

        if ok:
            out_df = pd.DataFrame([payload])
            out_df.to_csv(OUTPUT_CSV, index=False, mode='a', header=not header_written)
            header_written = True
            success_count += 1
        else:
            error_count += 1
            with open(ERRORS_CSV, "a", newline="") as ef:
                writer = csv.writer(ef)
                if os.path.getsize(ERRORS_CSV) == 0: writer.writerow(["row_index", "error", "url"])
                writer.writerow([idx, payload, url])

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(f"\nDone. Successes: {success_count}, Errors: {error_count}")

if __name__ == "__main__":
    if os.path.exists(INPUT_CSV):
        run_fetcher()
    else:
        print(f"Error: {INPUT_CSV} not found in current directory.")

In [ ]:
# Full Synchronous Open-Meteo Fetcher with 429 Protection and Chunking
!pip install -q requests pandas python-dateutil tqdm

import os, json, time, csv
from datetime import datetime, timezone
from dateutil import parser
import requests
import pandas as pd
from tqdm import tqdm
import bisect

# === CONFIG ===
INPUT_CSV = "enriched_collision_data_with_distance.csv"

# --- NEW CHUNKING CONFIG ---
START_ROW = 18100     # Change this to resume (e.g., set to 100 for the next batch)
LIMIT = 1000       # Set to None if you want to run to the end of the file

# Dynamically name output files so they NEVER fuse with existing files
end_marker = START_ROW + LIMIT if LIMIT else "end"
OUTPUT_CSV = f"weather_data_{START_ROW}_to_{end_marker}.csv"
ERRORS_CSV = f"errors_log_{START_ROW}_to_{end_marker}.csv"
# ---------------------------

RAW_RESP_DIR = "responses_raw"
REQUEST_TIMEOUT = 25
SLEEP_BETWEEN_REQUESTS = 0.0    # Gap between successful requests
MAX_RETRIES = 5                 # More retries for rate limit recovery
BACKOFF_FACTOR = 2.0

HOURLY_VARIABLES = [
    "temperature_2m", "relativehumidity_2m", "dewpoint_2m", "apparent_temperature",
    "precipitation", "rain", "showers", "snowfall", "snow_depth",
    "weathercode",
    "pressure_msl", "surface_pressure",
    "cloudcover", "cloudcover_low", "cloudcover_mid", "cloudcover_high",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation", "dni", "ghi", "gt", "gti",
    "windspeed_10m", "winddirection_10m", "windgusts_10m",
    "visibility",
    "evapotranspiration", "et0", "vapour_pressure_deficit",
    "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", "soil_temperature_54cm",
    "soil_moisture_0_1cm", "soil_moisture_1_3cm", "soil_moisture_3_9cm", "soil_moisture_9_27cm",
    "cape", "lifted_index", "is_day",
]

HOURLY_VARIABLES_2 = [
    "temperature_2m",
    "precipitation",
    "cloudcover",
    "windspeed_10m",
    "visibility",
]
os.makedirs(RAW_RESP_DIR, exist_ok=True)

# ---------------- HELPERS (Original Logic Preserved) ----------------

def filter_hourly_vars_for_endpoint(hourly_vars, endpoint):
    if "archive-api" in endpoint:
        allowed = {
            "temperature_2m", "relativehumidity_2m", "dewpoint_2m", "apparent_temperature",
            "precipitation", "rain", "showers", "snowfall", "snow_depth",
            "weathercode", "pressure_msl", "surface_pressure",
            "cloudcover", "cloudcover_low", "cloudcover_mid", "cloudcover_high",
            "windspeed_10m", "winddirection_10m", "windgusts_10m", "visibility"
        }
        return [v for v in hourly_vars if v in allowed]
    return hourly_vars

def _parse_datetime_from_row(row):
    for col in ("datetime", "timestamp", "dt"):
        if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
            try:
                dt = parser.isoparse(str(row[col]))
                if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
                return dt.astimezone(timezone.utc)
            except: pass

    def _get_int(cols_list, default=None):
        for c in cols_list:
            if c in row and pd.notna(row[c]) and str(row[c]).strip() != "":
                try: return int(float(str(row[c]).strip()))
                except: return default
        return default

    year, month, day = _get_int(["collision_year", "year"]), _get_int(["month"]), _get_int(["day"])
    if year is None or month is None or day is None: return None
    try:
        return datetime(year, month, day, _get_int(["hour"], 0), _get_int(["minutes","minute"], 0), 0, tzinfo=timezone.utc)
    except: return None

def _choose_endpoint_for_date(dt_utc):
    today_utc = datetime.now(timezone.utc).date()
    return "https://archive-api.open-meteo.com/v1/archive" if dt_utc.date() < today_utc else "https://api.open-meteo.com/v1/forecast"

def _request_open_meteo_once(lat, lon, dt_utc, hourly_vars):
    date_str = dt_utc.date().isoformat()
    endpoint = _choose_endpoint_for_date(dt_utc)
    filtered_vars = filter_hourly_vars_for_endpoint(hourly_vars, endpoint)
    params = {
        "latitude": float(lat), "longitude": float(lon),
        "start_date": date_str, "end_date": date_str,
        "hourly": ",".join(filtered_vars), "timezone": "UTC",
    }
    if endpoint.endswith("/forecast"): params["current_weather"] = "true"

    resp = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
    return resp

def request_with_retries(lat, lon, dt_utc, hourly_vars):
    attempt = 0
    wait = 2.0
    while attempt < MAX_RETRIES:
        try:
            resp = _request_open_meteo_once(lat, lon, dt_utc, hourly_vars)

            if resp.status_code == 429:
                pause_time = 60 if attempt < 2 else 300
                print(f"\n[!] Rate Limited (429). Sleeping {pause_time}s before retry {attempt+1}...")
                time.sleep(pause_time)
                attempt += 1
                continue

            resp.raise_for_status()
            return True, (resp.json(), resp.url)

        except Exception as e:
            print(f"\n[!] Connection error on row: {e}. Retrying in {wait}s...")
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, "Max retries exceeded or persistent 429"

def _find_best_hour_index(times_list, target_dt_utc):
    target_iso = target_dt_utc.strftime("%Y-%m-%dT%H:%M")
    try:
        idx = times_list.index(target_iso)
        return idx, "exact", times_list[idx]
    except ValueError:
        pos = bisect.bisect_left(times_list, target_iso)
        candidates = [p for p in [pos - 1, pos] if 0 <= p < len(times_list)]
        def iso_to_dt(s): return parser.isoparse(s).replace(tzinfo=timezone.utc)
        best = min(candidates, key=lambda i: abs((iso_to_dt(times_list[i]) - target_dt_utc).total_seconds()))
        return best, "nearest", times_list[best]

# ---------------- WORKER (Called Synchronously) ----------------

def _process_row_sync(idx, row):
    try:
        row = row.fillna("")
        lat, lon = row.get("latitude"), row.get("longitude")
        if not lat or not lon: return (idx, False, "missing lat/lon", None)

        dt_utc = _parse_datetime_from_row(row)
        if dt_utc is None: return (idx, False, "missing or unparseable datetime", None)

        ok, payload_or_err = request_with_retries(lat, lon, dt_utc, HOURLY_VARIABLES)
        if not ok: return (idx, False, f"request failed: {payload_or_err}", None)

        payload, url = payload_or_err

        try:
            with open(os.path.join(RAW_RESP_DIR, f"row_{idx}_raw.json"), "w") as fh:
                json.dump(payload, fh)
        except: pass

        hourly = payload.get("hourly", {})
        times = hourly.get("time")
        if not times: return (idx, False, "no hourly.time in response", url)

        idx_hour, match_type, returned_time = _find_best_hour_index(times, dt_utc)

        result_row = {col: row[col] for col in row.index}
        result_row.update({
            "_row_index": idx, "requested_datetime_utc": dt_utc.isoformat(),
            "returned_time_utc": returned_time, "match_type": match_type
        })

        for meta in ("latitude", "longitude", "generationtime_ms", "utc_offset_seconds"):
            if meta in payload: result_row[meta] = payload[meta]

        for var in HOURLY_VARIABLES:
            series = hourly.get(var)
            result_row[var] = series[idx_hour] if (series and idx_hour < len(series)) else None

        return (idx, True, result_row, url)
    except Exception as e:
        return (idx, False, f"unexpected error: {e}", None)

# ---------------- MAIN STREAMING LOOP ----------------

def run_fetcher():
    df = pd.read_csv(INPUT_CSV, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    # --- NEW SLICING LOGIC ---
    end_row = START_ROW + LIMIT if LIMIT else len(df)
    df_slice = df.iloc[START_ROW:end_row]
    # -------------------------

    if os.path.exists(OUTPUT_CSV): os.remove(OUTPUT_CSV)
    if os.path.exists(ERRORS_CSV): os.remove(ERRORS_CSV)

    header_written = False
    success_count, error_count = 0, 0

    # Iterating over the sliced DataFrame
    for i, row in tqdm(df_slice.iterrows(), total=len(df_slice), desc=f"Fetching Rows {START_ROW} to {end_row}"):
        idx, ok, payload, url = _process_row_sync(i, row)

        if ok:
            out_df = pd.DataFrame([payload])
            out_df.to_csv(OUTPUT_CSV, index=False, mode='a', header=not header_written)
            header_written = True
            success_count += 1
        else:
            error_count += 1
            with open(ERRORS_CSV, "a", newline="") as ef:
                writer = csv.writer(ef)
                if os.path.getsize(ERRORS_CSV) == 0: writer.writerow(["row_index", "error", "url"])
                writer.writerow([idx, payload, url])

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(f"\nDone. Saved to {OUTPUT_CSV}. Successes: {success_count}, Errors: {error_count}")

if __name__ == "__main__":
    if os.path.exists(INPUT_CSV):
        run_fetcher()
    else:
        print(f"Error: {INPUT_CSV} not found in current directory.")